# 30 · Part 3 - Demand flexibility and BTM battery opportunity (workspace)

Owner: ______  ·  Output: `outputs/part3_bess_candidates.csv`

Givens from the brief: persistent peak demand or limited capacity → good candidates; a 200 A service should not exceed **~3 kVA** of added load without further evaluation; the battery is 2 × Enphase IQ 10C (**20.16 kWh, 7.68 kW continuous, 11.52 kW 10-second peak, 89.4% round-trip**). The brief asks for impacts on peak demand, load shifting, capacity relief, bill savings, electrification enablement, and solar charging/discharging.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from avangrid.config import Paths, PROGRAM_BATTERY, INCREMENTAL_LOAD_LIMIT_KVA, SERVICE_KVA
from avangrid import pipeline, plots, labels, battery, features
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
P = Paths(ROOT)
D = pipeline.load_project(P)          # run 01_data_pipeline.ipynb once first
T = D.table
print(f"{len(T):,} customers | analysis-ready: {T.analysis_ready.sum():,}")
print(PROGRAM_BATTERY)
print(f"Service nameplate {SERVICE_KVA:.0f} kVA; incremental-load guideline {INCREMENTAL_LOAD_LIMIT_KVA} kVA")

25,653 customers | analysis-ready: 25,170
BatterySpec(units=2, energy_kwh_per_unit=10.08, continuous_kw_per_unit=3.84, peak_kw_per_unit=5.76, round_trip_efficiency=0.894, warranty_years=15)
Service nameplate 48 kVA; incremental-load guideline 3.0 kVA


## Engine reference
* `battery.simulate(load, request, spec=PROGRAM_BATTERY, reserve_frac=0.0, soc_init_frac=None, allow_export=False)`: `request` is what your strategy asks for each hour (+ discharge / − charge). The engine delivers what the limits allow. Arrays are `[customers, 8760]`.
* `battery.cap_request(load, cap, charge_allowed=None)`: request that holds demand at or below `cap` (scalar, per customer, per customer-day `[n, 365]`, or per hour).
* `battery.min_daily_cap(load, reserve_frac=0.0, discharge_hours=None, charge_hours=None)`: lowest daily cap `[n, 365]` the battery can hold *and* recharge within the day, with perfect foresight of that day (hour masks are bool `[24]`).
* `result.summary(eval_slots=...)`: peak before/after, kWh discharged/charged, losses, cycles, and demand at chosen hours (e.g. system peak hours).
* `battery.energy_cost(net, price_per_kwh)` and `battery.monthly_peak_kw(net, D.hours)`: tariff arithmetic for whatever rate you assume.
* For a one-unit option: `from avangrid.config import BatterySpec; one_unit = BatterySpec(units=1)`.

Example call pattern (not a strategy):

In [2]:
ids = T[T.analysis_ready].customer_number.sample(3, random_state=0).tolist()
L = D.matrix(ids)
night_h = np.isin(np.arange(24), range(0, 6))                    # example charge window (hour-of-day mask)
night = np.tile(night_h, 365)                                    # same mask over the 8760 hours
cap = battery.min_daily_cap(L, charge_hours=night_h)
res = battery.simulate(L, battery.cap_request(L, cap, charge_allowed=night), soc_init_frac=1.0)
res.summary(eval_slots=D.system.nsmallest(100, "rank").slot.to_numpy()).round(2).assign(customer=ids)

,peak_before_kw,peak_after_kw,kwh_discharged,kwh_charged,losses_kwh,equivalent_full_cycles,annual_kwh_before,annual_kwh_after,peak_reduction_kw,kw_eval_before,kw_eval_after,kw_eval_reduction,customer
0,6.67,2.35,519.72,568.99,60.95,27.27,3488.44,3537.71,4.31,1.64,1.10,0.54,7304
1,7.90,5.69,1085.71,1202.07,128.06,56.96,21842.04,21958.40,2.22,4.22,3.45,0.77,32164
2,8.21,3.94,1830.13,2033.39,216.25,96.01,11091.06,11294.32,4.27,3.40,2.90,0.50,7928


Note: the meter records grid **imports only**, so a solar home's midday surplus (what the battery could absorb) isn't directly visible. Only zero-import hours are.

## GridFlex v1 — physically constrained battery + illustrative DR

This section integrates `avangrid.gridflex` into the existing Part 3 project. Assumed tariffs, battery costs and system capacity values must be updated before investment decisions. The solver uses day-ahead perfect foresight with daily cyclic state of charge, so reported performance is an optimistic planning bound, not a guaranteed actual outcome.


In [ ]:
from avangrid.gridflex import GridFlexConfig, compare_scenarios, rank_customers, dispatch_year, demand_response
cfg = GridFlexConfig()
print(cfg)


## Critical system hours and sample scenario comparison

Hourly AMI kWh is interpreted as average kW in a one-hour interval. Top-100 system slots are an illustrative proxy for distribution capacity-critical hours, **not feeder-specific coincident peaks**.


In [ ]:
critical = np.zeros(len(D.hours), dtype=bool)
critical[D.system.nsmallest(100, "rank").slot.to_numpy(dtype=int)] = True
ids = T.loc[T.analysis_ready,"customer_number"].head(3).tolist()
example_id = ids[0]
load = D.series(example_id)
comparison = compare_scenarios(load, critical, D.hours, cfg)
display(comparison.round(2))


## Battery dispatch and SOC


In [ ]:
result = dispatch_year(load, critical, cfg=cfg)
fig, ax = plt.subplots(figsize=(12,4))
h = slice(0,168)
ax.plot(D.hours.ts.iloc[h], load[h], label="Baseline", alpha=.75)
ax.plot(D.hours.ts.iloc[h], result["net_kw"][h], label="Battery optimized", alpha=.75)
ax.set_ylabel("Hourly average kW"); ax.legend(); ax.set_title(f"Customer {example_id}: illustrative week")
plt.xticks(rotation=30);plt.tight_layout();plt.show()


## Candidate ranking and four-scenario export

Screens analysis-ready customers with the highest observed hourly peaks, then reports baseline, DR, battery, and combined deployment. This is screening, not final electrical service or financial approval. Set the number of candidates for the time available.


In [ ]:
cand = rank_customers(D, limit=30, cfg=cfg, critical_top=100, screen_method="critical_mean")
display(cand[cand.scenario == "battery_plus_dr"].head(15).round(2))
cand.to_csv(P.outputs / "part3_bess_candidates.csv", index=False)
print("Saved:", P.outputs / "part3_bess_candidates.csv")


## Critical caveats, electrification and solar

* **Electrification:** the challenge’s ~3 kVA incremental-load guideline is a screening threshold. A battery’s hourly kW reduction is **not** proof of compliance in kVA; service/feeder power factor, worst-hour coincidence and engineering approval are required.
* **PV:** net-import-only AMI does not reveal behind-the-meter PV generation, surplus, exported power, or PV charging. Our model charges from measured grid import; solar self-consumption benefits require PV generation/export or explicit separately labeled synthetic assumptions.
* **DR:** assumed shifts are energy-neutral, not inferred from actual flexible appliances or participation data.
* **Value accounting:** the critical-hour capacity value is a scenario assumption and may overlap with demand-charge or utility-program benefits. Avoid double counting.
* **Lifecycle:** no degradation, replacement, subsidy, tax, or interconnection cost is modeled by default. Backtest against held-out days and stress-test prices, CAPEX, SOC reserves, adoption and load growth.
